In [ ]:
# ==============================
# 📓 SysML Use Case Extraction Notebook
# ==============================

# Install dependencies (run once)
!pip install groq ipywidgets

import os
import json
from groq import Groq
from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Load API Key
# -----------------------------
client = Groq(api_key="GROQ_API_KEY")
MODEL = "llama-3.1-8b-instant"

# -----------------------------
# LLM Call
# -----------------------------
def call_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a systems engineering assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# -----------------------------
# Agent 1: Actor Extractor
# -----------------------------
def extract_actors(requirements):
    prompt = f"""
Extract all actors interacting with the system.

Rules:
- Include humans and external systems
- Exclude internal components
- Return JSON only

Requirements:
{requirements}

Output format:
{{
  "actors": []
}}
"""
    response = call_llm(prompt)
    return json.loads(response)

# -----------------------------
# Agent 2: Use Case Generator
# -----------------------------
def generate_use_cases(requirements, actors):
    prompt = f"""
Extract system use cases.

Steps:
1. Use the provided actors
2. Identify their goals
3. Convert each goal into a use case
4. Avoid duplicates

Actors:
{json.dumps(actors, indent=2)}

Requirements:
{requirements}

Output format:
{{
  "use_cases": [
    {{
      "name": "",
      "actor": "",
      "description": "",
      "preconditions": [],
      "postconditions": []
    }}
  ]
}}
"""
    response = call_llm(prompt)
    return json.loads(response)

# -----------------------------
# Agent 3: Validator
# -----------------------------
def validate_use_cases(requirements, actors, use_cases):
    prompt = f"""
Validate and improve the use cases.

Tasks:
1. Add missing use cases
2. Remove duplicates
3. Ensure all interactions are covered
4. Ensure correct actor mapping

Actors:
{json.dumps(actors, indent=2)}

Use Cases:
{json.dumps(use_cases, indent=2)}

Requirements:
{requirements}

Return ONLY final JSON:
{{
  "use_cases": []
}}
"""
    response = call_llm(prompt)
    return json.loads(response)

# -----------------------------
# Pipeline
# -----------------------------
def run_pipeline(requirements):
    actors = extract_actors(requirements)
    use_cases = generate_use_cases(requirements, actors)
    final_use_cases = validate_use_cases(requirements, actors, use_cases)

    return {
        "actors": actors["actors"],
        "use_cases": final_use_cases["use_cases"]
    }

# ==============================
# 🎛️ UI Components
# ==============================

# Text input
requirements_input = widgets.Textarea(
    value="",
    placeholder="Paste your system requirements here...",
    description="Requirements:",
    layout=widgets.Layout(width="100%", height="200px")
)

# File upload
file_upload = widgets.FileUpload(
    accept='.txt,.md',
    multiple=False,
    description="Upload File"
)

# Run button
run_button = widgets.Button(
    description="Run Extraction",
    button_style='success'
)

# Output area
output_area = widgets.Output()

# -----------------------------
# File Upload Handler
# -----------------------------
def handle_file_upload(change):
    if file_upload.value:
        uploaded_file = list(file_upload.value.values())[0]
        content = uploaded_file['content'].decode('utf-8')
        requirements_input.value = content

file_upload.observe(handle_file_upload, names='value')

# -----------------------------
# Run Button Handler
# -----------------------------
def on_run_clicked(b):
    with output_area:
        clear_output()
        print("🚀 Running pipeline...\n")

        requirements = requirements_input.value

        try:
            result = run_pipeline(requirements)
            print("✅ Extraction Complete!\n")
            print(json.dumps(result, indent=2))
        except Exception as e:
            print("❌ Error:", str(e))

run_button.on_click(on_run_clicked)

# -----------------------------
# Display UI
# -----------------------------
display(
    widgets.VBox([
        widgets.HTML("<h2>SysML Use Case Extraction Agent</h2>"),
        file_upload,
        requirements_input,
        run_button,
        output_area
    ])
)
